In [1]:
from io import StringIO
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler
from yellowbrick.cluster import KElbowVisualizer

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.transaction_ml_pipeline import (
    CLUSTERED_DATASET_PATH,
    FIGURES_DIR,
    PREPROCESSED_DATASET_PATH,
    RAW_DATASET_PATH,
    SUBMISSION_CLUSTER_DATA,
    SUBMISSION_CLUSTER_MODEL,
    KAGGLE_DATASET_SLUG,
    stage_kaggle_dataset,
)

In [2]:
print("Dataset source:", KAGGLE_DATASET_SLUG)
stage_kaggle_dataset()
df = pd.read_csv(RAW_DATASET_PATH)
df.head()

Dataset source: aryan208/financial-transactions-dataset-for-fraud-detection


,TransactionID,TransactionDate,AccountID,CounterpartyAccount,TransactionAmount,TransactionType,MerchantCategory,Location,DeviceType,IsFraud,FraudType,TimeSinceLastTransaction,SpendingDeviationScore,VelocityScore,GeoAnomalyScore,PaymentChannel,IPAddress,DeviceID
0,T100000,2023-08-22T09:22:43.516168,ACC877572,ACC388389,343.78,withdrawal,utilities,Tokyo,mobile,False,NaN,NaN,-0.21,3,0.22,card,13.101.214.112,D8536477
1,T100001,2023-08-04T01:58:02.606711,ACC895667,ACC944962,419.65,withdrawal,online,Toronto,atm,False,NaN,NaN,-0.14,7,0.96,ACH,172.52.47.194,D2622631
2,T100002,2023-05-12T11:39:33.742963,ACC733052,ACC377370,2773.86,deposit,other,London,pos,False,NaN,NaN,-1.78,20,0.89,card,185.98.35.23,D4823498
3,T100003,2023-10-10T06:04:43.195112,ACC996865,ACC344098,1666.22,deposit,online,Sydney,pos,False,NaN,NaN,-0.60,6,0.37,wire_transfer,107.136.36.87,D9961380
4,T100004,2023-09-24T08:09:02.700162,ACC584714,ACC497887,24.43,transfer,utilities,Toronto,mobile,False,NaN,NaN,0.79,13,0.27,ACH,108.161.108.255,D7637601


In [3]:
buffer = StringIO()
df.info(buf=buffer)
print(buffer.getvalue())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   TransactionID             30000 non-null  object 
 1   TransactionDate           30000 non-null  object 
 2   AccountID                 30000 non-null  object 
 3   CounterpartyAccount       30000 non-null  object 
 4   TransactionAmount         30000 non-null  float64
 5   TransactionType           30000 non-null  object 
 6   MerchantCategory          30000 non-null  object 
 7   Location                  30000 non-null  object 
 8   DeviceType                30000 non-null  object 
 9   IsFraud                   30000 non-null  bool   
 10  FraudType                 11 non-null     object 
 11  TimeSinceLastTransaction  496 non-null    float64
 12  SpendingDeviationScore    30000 non-null  float64
 13  VelocityScore             30000 non-null  int64  
 14  GeoAno

In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
TransactionID,30000,30000,T100000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TransactionDate,30000,30000,2023-08-22T09:22:43.516168,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AccountID,30000,29504,ACC822717,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CounterpartyAccount,30000,29536,ACC118520,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TransactionAmount,30000.0,NaN,NaN,NaN,356.756896,466.740467,0.01,26.43,135.695,499.245,2773.86
TransactionType,30000,4,transfer,7616,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MerchantCategory,30000,8,other,3864,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,30000,8,Tokyo,3902,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DeviceType,30000,4,web,7561,NaN,NaN,NaN,NaN,NaN,NaN,NaN
IsFraud,30000,2,False,29989,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.isnull().sum()

TransactionID                   0
TransactionDate                 0
AccountID                       0
CounterpartyAccount             0
TransactionAmount               0
TransactionType                 0
MerchantCategory                0
Location                        0
DeviceType                      0
IsFraud                         0
FraudType                   29989
TimeSinceLastTransaction    29504
SpendingDeviationScore          0
VelocityScore                   0
GeoAnomalyScore                 0
PaymentChannel                  0
IPAddress                       0
DeviceID                        0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
prepared_df = df.drop(columns=["FraudType", "TimeSinceLastTransaction"], errors="ignore")
cleaned_df = prepared_df.dropna().drop_duplicates().copy()
cleaned_df = cleaned_df.drop(
    columns=[
        "TransactionID",
        "AccountID",
        "CounterpartyAccount",
        "DeviceID",
        "IPAddress",
        "TransactionDate",
        "IsFraud",
    ],
    errors="ignore",
)
cleaned_df.head()

,TransactionAmount,TransactionType,MerchantCategory,Location,DeviceType,SpendingDeviationScore,VelocityScore,GeoAnomalyScore,PaymentChannel
0,343.78,withdrawal,utilities,Tokyo,mobile,-0.21,3,0.22,card
1,419.65,withdrawal,online,Toronto,atm,-0.14,7,0.96,ACH
2,2773.86,deposit,other,London,pos,-1.78,20,0.89,card
3,1666.22,deposit,online,Sydney,pos,-0.60,6,0.37,wire_transfer
4,24.43,transfer,utilities,Toronto,mobile,0.79,13,0.27,ACH


In [8]:
encoded_df = cleaned_df.copy()
encoders = {}
for column in encoded_df.select_dtypes(include="object").columns:
    encoder = LabelEncoder()
    encoded_df[column] = encoder.fit_transform(encoded_df[column].astype(str))
    encoders[column] = encoder

encoded_df.to_csv(PREPROCESSED_DATASET_PATH, index=False)
encoded_df.head()

,TransactionAmount,TransactionType,MerchantCategory,Location,DeviceType,SpendingDeviationScore,VelocityScore,GeoAnomalyScore,PaymentChannel
0,343.78,3,7,6,1,-0.21,3,0.22,2
1,419.65,3,2,7,0,-0.14,7,0.96,0
2,2773.86,0,3,2,2,-1.78,20,0.89,2
3,1666.22,0,2,5,2,-0.60,6,0.37,3
4,24.43,2,7,7,1,0.79,13,0.27,0


In [9]:
scaler = StandardScaler()
scaled_df = pd.DataFrame(scaler.fit_transform(encoded_df), columns=encoded_df.columns)
scaled_df.head()

,TransactionAmount,TransactionType,MerchantCategory,Location,DeviceType,SpendingDeviationScore,VelocityScore,GeoAnomalyScore,PaymentChannel
0,-0.027804,1.351387,1.521203,1.087041,-0.450874,-0.210076,-1.298980,-0.982700,0.433838
1,0.134752,1.351387,-0.661047,1.524341,-1.343929,-0.140220,-0.604988,1.597279,-1.353835
2,5.178774,-1.339487,-0.224597,-0.662160,0.442182,-1.776850,1.650488,1.353227,0.433838
3,2.805595,-1.339487,-0.661047,0.649741,0.442182,-0.599275,-0.778486,-0.459731,1.327675
4,-0.712028,0.454429,1.521203,1.524341,-0.450874,0.787869,0.436001,-0.808377,-1.353835


In [10]:
elbow_model = KMeans(random_state=42, n_init=20)
elbow_model._estimator_type = "clusterer"
visualizer = KElbowVisualizer(elbow_model, k=(2, 8), timings=False)
visualizer.fit(scaled_df)
visualizer.show(outpath=str(FIGURES_DIR / "notebook_elbow_method.png"))
best_k = visualizer.elbow_value_
if best_k is None:
    best_k = 2
best_k

C:\Users\Muh Fajri\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Muh Fajri\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Muh Fajri\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Muh Fajri\AppData\Local\Programs\Python\Python311\Lib\s

C:\Users\Muh Fajri\AppData\Local\Programs\Python\Python311\Lib\site-packages\yellowbrick\utils\kneed.py:156: YellowbrickWarning: No 'knee' or 'elbow point' detected This could be due to bad clustering, no actual clusters being formed etc.
  warnings.warn(warning_message, YellowbrickWarning)
C:\Users\Muh Fajri\AppData\Local\Programs\Python\Python311\Lib\site-packages\yellowbrick\cluster\elbow.py:374: YellowbrickWarning: No 'knee' or 'elbow' point detected, pass `locate_elbow=False` to remove the warning
  warnings.warn(warning_message, YellowbrickWarning)


2

In [11]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(scaled_df)
joblib.dump(kmeans, SUBMISSION_CLUSTER_MODEL)

['D:\\code\\Machine Learning\\BMLP_Muhammad-Fajri\\model_clustering.h5']

In [12]:
clustered_df = encoded_df.copy()
clustered_df["Target"] = cluster_labels
clustered_df.to_csv(CLUSTERED_DATASET_PATH, index=False)
clustered_df.to_csv(SUBMISSION_CLUSTER_DATA, index=False)
clustered_df.head()

,TransactionAmount,TransactionType,MerchantCategory,Location,DeviceType,SpendingDeviationScore,VelocityScore,GeoAnomalyScore,PaymentChannel,Target
0,343.78,3,7,6,1,-0.21,3,0.22,2,1
1,419.65,3,2,7,0,-0.14,7,0.96,0,1
2,2773.86,0,3,2,2,-1.78,20,0.89,2,0
3,1666.22,0,2,5,2,-0.60,6,0.37,3,0
4,24.43,2,7,7,1,0.79,13,0.27,0,1


In [13]:
cleaned_with_target = cleaned_df.copy()
cleaned_with_target["Target"] = cluster_labels
numeric_columns = cleaned_with_target.select_dtypes(include="number").columns.drop("Target")
cluster_summary = cleaned_with_target.groupby("Target")[numeric_columns].agg(["mean", "min", "max"])
cluster_summary

TransactionAmount                  SpendingDeviationScore              \
                    mean     min      max                   mean   min   max   
Target                                                                         
0            1058.161989  209.32  2773.86                0.00031 -3.65  3.39   
1             141.971246    0.01   990.24                0.00057 -4.15  4.09   

       VelocityScore         GeoAnomalyScore            
                mean min max            mean  min  max  
Target                                                  
0          10.479738   1  20        0.496892  0.0  1.0  
1          10.489224   1  20        0.503384  0.0  1.0

In [14]:
cleaned_with_target.groupby("Target").agg(
    transaction_count=("TransactionAmount", "count"),
    avg_transaction_amount=("TransactionAmount", "mean"),
    avg_spending_deviation=("SpendingDeviationScore", "mean"),
    avg_velocity_score=("VelocityScore", "mean"),
    avg_geo_anomaly_score=("GeoAnomalyScore", "mean"),
)

,transaction_count,avg_transaction_amount,avg_spending_deviation,avg_velocity_score,avg_geo_anomaly_score
Target,,,,,
0,7033,1058.161989,0.00031,10.479738,0.496892
1,22967,141.971246,0.00057,10.489224,0.503384
